In [ ]:
import torch
import torch.nn as nn
from torch.distributions import Categorical
import numpy as np
import torch.nn.functional as F


In [ ]:

class HyperparameterController(nn.Module):
    def __init__(self, pos_len, pos_size):
        super(HyperparameterController, self).__init__()

        self.pos_len = pos_len
        self.pos_size = pos_size
        self.max_layers = max(pos_len) # Defines how many independent size heads we need

        NNmid = 38
        self.shared1 = nn.Linear(1, NNmid)
        self.shared2 = nn.Linear(NNmid, NNmid)
        self.gaussian = GaussianActivation()

        nn.init.normal_(self.shared1.weight, mean=0.0, std=0.1)
        nn.init.zeros_(self.shared1.bias)
        nn.init.normal_(self.shared2.weight, mean=0.0, std=0.1)
        nn.init.zeros_(self.shared2.bias)

        # Standard independent heads
        self.act_head = nn.Linear(NNmid, 4)
        self.lr_head = nn.Linear(NNmid, 4)

        # Dynamically sized heads based on input lists
        self.layers_head = nn.Linear(NNmid, len(self.pos_len))
        self.size_heads = nn.ModuleList([nn.Linear(NNmid, len(self.pos_size)) for _ in range(self.max_layers)])

        # Mapping dictionaries
        self.act_map = {0: 'relu', 1: 'sigmoid', 2: 'softmax', 3: 'gauss'}
        self.lr_map = {0: 0.1, 1: 0.5, 2: 1.0, 3: 2.0}

        # Dynamic mapping dictionaries
        self.layers_map = {i: val for i, val in enumerate(self.pos_len)}
        self.size_map = {i: val for i, val in enumerate(self.pos_size)}

    def forward(self, x):
        x1 = self.gaussian(self.shared1(x))
        x2 = self.gaussian(self.shared2(x1))

        act_logits = self.act_head(x2)
        lr_logits = self.lr_head(x2)
        layers_logits = self.layers_head(x2)

        # Get logits for each independent layer size
        size_logits_list = [head(x2) for head in self.size_heads]

        return act_logits, lr_logits, layers_logits, size_logits_list

    def sample_hyperparameters(self):
        dummy_input = torch.ones(1)
        act_logits, lr_logits, layers_logits, size_logits_list = self.forward(dummy_input)

        act_dist = Categorical(logits=act_logits)
        lr_dist = Categorical(logits=lr_logits)
        layers_dist = Categorical(logits=layers_logits)

        act_idx = act_dist.sample()
        lr_idx = lr_dist.sample()
        layers_idx = layers_dist.sample()

        log_prob = act_dist.log_prob(act_idx) + lr_dist.log_prob(lr_idx) + layers_dist.log_prob(layers_idx)

        num_layers = self.layers_map[layers_idx.item()]

        layer_sizes = []
        for i in range(num_layers):
            size_dist = Categorical(logits=size_logits_list[i])
            size_idx = size_dist.sample()

            log_prob += size_dist.log_prob(size_idx)
            layer_sizes.append(self.size_map[size_idx.item()])

        params = {
            'activation': self.act_map[act_idx.item()],
            'learning_rate': self.lr_map[lr_idx.item()],
            'num_layers': num_layers,
            'layer_sizes': layer_sizes
        }

        return params, log_prob


In [ ]:
def train_child_network(params, x_train, y_train, K, N_epochs=1000, cut_per=0.25):
    mid_size_array = params['layer_sizes']

    model, x_train_tensor, y_onehot, optimizer, cost_func = setup_model_and_data(
        x_train, y_train, K,
        mid_size_array,
        params['learning_rate'],
        N_epochs,
        params['activation']
    )

    cost_history = []
    acc_history = []

    for epoch in range(N_epochs):
        optimizer.zero_grad()
        NN_output = model(x_train_tensor)
        cost = cost_func(NN_output, y_onehot)
        cost.backward()
        optimizer.step()

        cost_history.append(cost.item())

        predicted_class = np.argmax(NN_output.detach().numpy(), axis=1)
        accuracy = np.mean(predicted_class == y_train)
        acc_history.append(accuracy)

    reward = rewardFN(acc_history, cost_history, cut_per)
    return reward

In [ ]:

class NASOptimizer:
    def __init__(self, pos_len, pos_size, controller_lr=0.01):
        # Pass the lists into the controller upon instantiation
        self.controller = HyperparameterController(pos_len, pos_size)
        self.optimizer = torch.optim.Adam(self.controller.parameters(), lr=controller_lr)

    def optimize(self, x_train, y_train, K, search_episodes=50, N_epochs=1000):
        #initialize exponentially weighted moving average
        baseline_reward = 0.0
        print("Starting Neural Architecture Search...")

        for episode in range(search_episodes):
            self.optimizer.zero_grad()

            params, log_prob = self.controller.sample_hyperparameters()
            reward = train_child_network(params, x_train, y_train, K, N_epochs)

            if episode == 0:
                baseline_reward = reward
            else:
                #update exponentially weighted moving average
                baseline_reward = 0.9 * baseline_reward + 0.1 * reward

            #use relative advantage from last run to determine gradient
            advantage = reward - baseline_reward

            #algorithm minimizes loss, want to maximize certainty * advantage
            loss = -log_prob * advantage
            loss.backward()
            self.optimizer.step()

            print(f"Episode {episode+1:02d} | Reward: {reward:.4f} | Params: {params}")

        print("\nSearch Complete. Evaluating optimal configuration...")

        with torch.no_grad():
            #initialize with ones, statis environment, so doesn't matter
            dummy_input = torch.ones(1)
            act_logits, lr_logits, layers_logits, size_logits_list = self.controller(dummy_input)

            #put in temperature to widen search results, temp=1 is no temperature
            temp=2.0
            act_logits = act_logits/temp
            lr_logits = lr_logits/temp
            layers_logits = layers_logits/temp
            size_logits_list = [logits/temp for logits in size_logits_list]

            #normalize raw logits to probability distributions
            self.act_probs = F.softmax(act_logits, dim=-1).numpy()
            self.lr_probs = F.softmax(lr_logits, dim=-1).numpy()
            self.layers_probs = F.softmax(layers_logits, dim=-1).numpy()
            self.size_probs_list = [F.softmax(logits, dim=-1).numpy() for logits in size_logits_list]

            #get num of layers and size of each
            num_layers = self.controller.layers_map[torch.argmax(layers_logits).item()]
            best_layer_sizes = [self.controller.size_map[torch.argmax(size_logits_list[i]).item()] for i in range(num_layers)]

            #pass best parameters
            self.best_params = {
                'activation': self.controller.act_map[torch.argmax(act_logits).item()],
                'learning_rate': self.controller.lr_map[torch.argmax(lr_logits).item()],
                'num_layers': num_layers,
                'layer_sizes': best_layer_sizes
            }

        print(f"Optimal Hyperparameters Discovered: {self.best_params}")

        return self.best_params, self.act_probs, self.lr_probs, self.layers_probs, self.size_probs_list